# Preprocessing and Validation of the Wi-Fi Fingerprinting Model
This notebook processes Wi-Fi signal data (the *radio map*), standardizes it, and evaluates the **K-Nearest Neighbors (KNN)** algorithm. The model is validated with **Group K-Fold** and **Leave-One-Group-Out (LOGO)**. Once validated, the final model is saved.

## 1. Imports & Configuration

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib  # For saving the model
import re
from pathlib import Path

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Configuration - every path is resolved relative to the repository root,
# so the notebook works from any machine and no personal paths are stored.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements.txt').exists()),
    Path.cwd(),
)
BASE_DIR = str(PROJECT_ROOT / 'data' / 'raw')              # raw CSV scans (git-ignored)
PROCESSED_DIR = str(PROJECT_ROOT / 'data' / 'processed')   # intermediate datasets (git-ignored)
MODEL_DIR = str(PROJECT_ROOT / 'models')                   # trained model + scaler (git-ignored)
DEVICES = ['device_1', 'device_2', 'device_3', 'device_4']
DEFAULT_DBM = -100

# Site-specific values (SSID keyword, access point BSSIDs) are NOT hard-coded here.
# They are read from the private config; the public example is used as a fallback.
AP_CONFIG_PATH = Path(os.getenv('IPS_AP_CONFIG', str(PROJECT_ROOT / 'config' / 'access_points.private.json')))
if not AP_CONFIG_PATH.exists():
    AP_CONFIG_PATH = PROJECT_ROOT / 'config' / 'access_points.example.json'
with open(AP_CONFIG_PATH, encoding='utf-8') as f:
    AP_CONFIG = json.load(f)
SSID_KEYWORD = AP_CONFIG['ssid_keyword']
print(f'Using access point config: {AP_CONFIG_PATH.name}')


## 2. Collecting and Parsing the CSV Data

In [ ]:
def extract_fingerprint_to_dict(filepath, col_wifi, col_mac, col_dbm):
    try:
        df = pd.read_csv(filepath)
    except Exception:
        return {}

    df_filtered = df[df[col_wifi].astype(str).str.lower().str.contains(SSID_KEYWORD, na=False)].copy()
    
    if not df_filtered.empty:
        df_filtered['mac_normalized'] = (
            df_filtered[col_mac]
            .astype(str)
            .str.lower()
            .str.replace('-', ':')
            .str.rstrip(':') 
        )
        return dict(zip(df_filtered['mac_normalized'], df_filtered[col_dbm]))
    return {}

# 1. Collect all data into a list of dictionaries
all_records = []

for device in DEVICES:
    dir_dev = os.path.join(BASE_DIR, device)
    if not os.path.exists(dir_dev):
        continue
        
    for filepath in glob.glob(os.path.join(dir_dev, '*.csv')):
        filename = os.path.basename(filepath).replace('.csv', '')
        
        # File names follow one convention (e.g. room1_1), so they can be lowercased directly
        clean_name = filename.lower()
        
        # Column names depend on the device that produced the CSV
        if device == 'device_1':
            col_wifi = 'SSID'
            col_mac = 'Target MAC Address'
            col_dbm = 'Sinyal Terbaru (dBm)'  # raw column name exported by the device_1 logger
        else:
            col_wifi = 'Name'
            col_mac = 'Address'
            col_dbm = 'Signal Level'
            
        # Split the location (room) from the reference point (e.g. 'room1' and '1')
        try:
            location, point = clean_name.split('_', 1)
        except ValueError:
            continue  # Skip files whose name does not follow the convention
            
        # Get the signal data {mac_address: dbm}
        signal_dict = extract_fingerprint_to_dict(filepath, col_wifi, col_mac, col_dbm)
        
        # If any signal was captured, attach the metadata and store the record
        if signal_dict:
            signal_dict['reference_point'] = point
            signal_dict['location'] = location.upper() 
            signal_dict['device'] = device
            
            all_records.append(signal_dict)


In [ ]:
# Preview the first few records
for record in all_records[:5]:  # Show the first 5 records
    print(record)
    
# Save all_records as a text file
os.makedirs(PROCESSED_DIR, exist_ok=True)
output_path = os.path.join(PROCESSED_DIR, 'fingerprint_data.txt')
with open(output_path, 'w') as f:
    for record in all_records:
        f.write(str(record) + '\n')
print(f"Fingerprint data saved to {output_path}")

## 3. DataFrame Preprocessing (Cleaning & Sorting)

In [ ]:
# 1. Allowed MAC addresses (whitelist), taken from the access point config
list_ap = list(AP_CONFIG['access_points'].keys())
list_ap = [mac.lower() for mac in list_ap]

# 2. Convert to a Pandas DataFrame
df_radiomap = pd.DataFrame(all_records)

# 3. Tidy up the DataFrame
if not df_radiomap.empty:
    # Fill NaN values with the default dBm value (-100)
    df_radiomap.fillna(DEFAULT_DBM, inplace=True)
    
    metadata_cols = ['reference_point', 'location', 'device']
            
    # Lock mac_cols so its contents and order match list_ap exactly
    mac_cols = list_ap
    
    # Make sure the MAC columns are numeric (float) for the model
    df_radiomap[mac_cols] = df_radiomap[mac_cols].astype(float)
    
    # Reassemble the column order.
    # This automatically drops every "stray" MAC address that is not in list_ap
    df_radiomap = df_radiomap[mac_cols + metadata_cols]
    
    # Sort rows by device, location, then reference point
    df_radiomap['sort_point'] = pd.to_numeric(df_radiomap['reference_point'], errors='coerce')
    df_radiomap = df_radiomap.sort_values(by=['device', 'location', 'sort_point']).drop(columns=['sort_point']).reset_index(drop=True)

dataset = df_radiomap
dataset["reference_point"] = dataset["reference_point"].astype(int)


## 4. Data Inspection (EDA)

In [ ]:
display(dataset.head())

print("\n=== Dataset Info ===")
dataset.info()

print("\n=== Number of Reference Points per Location ===")
print(dataset.groupby("location")["reference_point"].nunique())

# Save the processed dataset as CSV for the modelling stage
processed_csv_path = os.path.join(PROCESSED_DIR, 'processed_fingerprint_data.csv')
dataset.to_csv(processed_csv_path, index=False)
print(f"Processed dataset saved to {processed_csv_path}")


## 5. Initial Experiment: Manual Split & Hyperparameter Tuning (KNN)
Find the best value of $K$ using a manual split (train: points 1-5, validation: points 6-9).

In [ ]:
train_df = dataset[dataset["reference_point"].between(1, 5)]
valid_df = dataset[dataset["reference_point"].between(6, 9)]

print("Training samples:", len(train_df))
print("Validation samples:", len(valid_df))

X_train = train_df.drop(columns=["reference_point", "location", "device"]).fillna(DEFAULT_DBM)
y_train = train_df["location"]

X_valid = valid_df.drop(columns=["reference_point", "location", "device"]).fillna(DEFAULT_DBM)
y_valid = valid_df["location"]

# Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Preview the scaled data
print("\nPreview X_train_scaled:")
print(X_train_scaled[:5])  # First 5 rows of the scaled training data
print("\nPreview X_valid_scaled:")
print(X_valid_scaled[:5])  # First 5 rows of the scaled validation data

# Save the scaled data as CSV files for reference or further analysis
scaled_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
scaled_train_df['location'] = y_train.values
scaled_valid_df = pd.DataFrame(X_valid_scaled, columns=X_valid.columns)
scaled_valid_df['location'] = y_valid.values
scaled_train_df.to_csv(os.path.join(PROCESSED_DIR, 'scaled_train_data.csv'), index=False)
scaled_valid_df.to_csv(os.path.join(PROCESSED_DIR, 'scaled_valid_data.csv'), index=False)

In [ ]:
# Evaluate the optimal K
print("\nK evaluation results (odd values 1-15):")
for k in range(1, 16, 2):
    knn = KNeighborsClassifier(
        n_neighbors=k,
        weights='distance',
        metric='manhattan'
    )
    knn.fit(X_train_scaled, y_train)
    acc = knn.score(X_valid_scaled, y_valid)
    print(f"K={k:02d} | Accuracy={acc:.4f}")


In [ ]:
# Sort strings so that numbers follow human logic (natural sorting). Example: room2 comes before room12
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', str(s))]

# Choose the best K whose confusion matrix we want to inspect
best_k = 11 

# 1. Initialize and retrain the model with a single K value
knn_best = KNeighborsClassifier(
    n_neighbors=best_k,
    weights='distance',
    metric='manhattan'
)
knn_best.fit(X_train_scaled, y_train)

# 2. Predict on the validation data
y_pred_best = knn_best.predict(X_valid_scaled)

# 3. Compute and display the confusion matrix
labels = sorted(y_valid.unique(), key=natural_sort_key)
cm = confusion_matrix(y_valid, y_pred_best, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Absolute (number of actual predictions)
disp_abs = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp_abs.plot(cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion Matrix - Absolute (K={best_k})")
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Percentage (per-location accuracy proportion)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp_pct = ConfusionMatrixDisplay(confusion_matrix=cm_percent, display_labels=labels)
disp_pct.plot(cmap="Blues", ax=axes[1], values_format='.2f')
axes[1].set_title(f"Confusion Matrix - Percentage (K={best_k})")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Classification report for the best K
from sklearn.metrics import classification_report
print(f"\nClassification Report for K={best_k}:\n")
print(classification_report(y_valid, y_pred_best, target_names=labels))

## 6. Cross-Validation: Data Setup
Initialize `X`, `y`, and the group labels before running *Group K-Fold* and *LOGO*.

In [ ]:
X = dataset.drop(columns=["reference_point", "location", "device"]).fillna(DEFAULT_DBM)
y = dataset["location"]
groups = dataset["reference_point"]

labels = sorted(y.unique(), key=natural_sort_key)


### 6.1. Group K-Fold Cross-Validation

In [ ]:
gkf = GroupKFold(n_splits=5)

scaler_gkf = StandardScaler()

accuracies_gkf = []
y_true_gkf = []
y_pred_gkf = []

print("=== Group K-Fold Cross Validation ===\n")

for fold_idx, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    x_train_scaled = scaler_gkf.fit_transform(X_train)
    x_valid_scaled = scaler_gkf.transform(X_valid)

    knn = KNeighborsClassifier(
        n_neighbors=11,
        weights='distance',
        metric='manhattan'
    )

    knn.fit(x_train_scaled, y_train)
    y_pred = knn.predict(x_valid_scaled)

    acc = accuracy_score(y_valid, y_pred)
    accuracies_gkf.append(acc)
    
    y_true_gkf.extend(y_valid)
    y_pred_gkf.extend(y_pred)

    print(f"[Fold {fold_idx}]")
    print(f"Train groups: {groups.iloc[train_idx].unique()}")
    print(f"Valid groups: {groups.iloc[valid_idx].unique()}")
    print(f"Accuracy    : {acc:.4f}\n")

print("Accuracy per fold :", [f"{a:.4f}" for a in accuracies_gkf])
print("Mean accuracy     :", f"{np.mean(accuracies_gkf):.4f}")


**Group K-Fold Confusion Matrix**

In [ ]:
cm_gkf = confusion_matrix(y_true_gkf, y_pred_gkf, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp_gkf = ConfusionMatrixDisplay(confusion_matrix=cm_gkf, display_labels=labels)
disp_gkf.plot(cmap="Blues", ax=ax)

ax.set_title("Confusion Matrix - Group K-Fold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### 6.2. Leave-One-Group-Out (LOGO) Cross-Validation

In [ ]:
logo = LeaveOneGroupOut()

scaler_logo = StandardScaler()

accuracies_logo = []
y_true_logo = []
y_pred_logo = []

print("=== Leave-One-Group-Out (LOGO) Cross Validation ===\n")

for train_idx, valid_idx in logo.split(X, y, groups):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    X_train_scaled = scaler_logo.fit_transform(X_train)
    X_valid_scaled = scaler_logo.transform(X_valid)
    
    # KNN model with K=11 (best value from the earlier K evaluation)
    knn = KNeighborsClassifier(
        n_neighbors=11,
        weights='distance',
        metric='manhattan'
    )
    
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_valid_scaled)
    
    acc = accuracy_score(y_valid, y_pred)
    accuracies_logo.append(acc)
    
    y_true_logo.extend(y_valid)
    y_pred_logo.extend(y_pred)

print("Accuracy per reference point :", [f"{a:.4f}" for a in accuracies_logo])
print("Mean accuracy (LOGO)         :", f"{np.mean(accuracies_logo):.4f}")


**LOGO Confusion Matrix**

In [ ]:
cm_logo = confusion_matrix(y_true_logo, y_pred_logo, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
disp_logo = ConfusionMatrixDisplay(confusion_matrix=cm_logo, display_labels=labels)
disp_logo.plot(cmap="Blues", ax=ax)

ax.set_title("Confusion Matrix - LOGO")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Training and Saving the Final Model
Once the cross-validation results are satisfactory, the best practice is to retrain the model on the **entire dataset**. The final model and its `StandardScaler` are saved with `joblib` so they are ready for *deployment* or real-time prediction (see `app/app.py`).

In [ ]:
# Separate features and labels from the full dataset
X_final = dataset.drop(columns=["reference_point", "location", "device"]).fillna(DEFAULT_DBM)
y_final = dataset["location"]

# Create the models folder if it does not exist yet
model_dir = MODEL_DIR
os.makedirs(model_dir, exist_ok=True)

# 1. Initialize and fit the scaler
final_scaler = StandardScaler()
X_final_scaled = final_scaler.fit_transform(X_final)

# 2. Initialize and fit the KNN model
# Use the best K found during evaluation (K=11)
final_knn = KNeighborsClassifier(
    n_neighbors=11,
    weights='distance',
    metric='manhattan'
)
final_knn.fit(X_final_scaled, y_final)

# 3. Save the model and the scaler
model_path = os.path.join(model_dir, 'knn_model_radiomap.pkl')
scaler_path = os.path.join(model_dir, 'scaler_radiomap.pkl')

joblib.dump(final_knn, model_path)
joblib.dump(final_scaler, scaler_path)

print("=== Save complete ===")
print(f"Model saved to : {model_path}")
print(f"Scaler saved to: {scaler_path}")
